# Lab Day 10: Finetune Qwen3-VL-4B (LoRA) để định vị hư hại mặt đường
Notebook chạy mã trong `student/`; GPU dùng cho train/inference, checkpoints nên lưu trên Drive. Đọc `README.md` và `GUIDE.md` trước. Không ghi API key hoặc đưa dữ liệu/trọng số lên Git.

## 0. Môi trường (Colab T4)

In [ ]:
# Mount Drive ?? ??c bundle m? v? l?u checkpoint khi train.
from google.colab import drive
import zipfile, os
drive.mount('/content/drive')
with zipfile.ZipFile('/content/drive/MyDrive/day10_source_bundle.zip') as zf:
    zf.extractall('/content/day10')
%cd /content/day10
print('Source ready:', os.path.isdir('student'), os.path.isdir('scripts'))


In [ ]:
# Runtime → Change runtime type → T4 GPU
# Make the updated repository files available in /content/lab_day10 before running this notebook.
# For a GitHub checkout, open this notebook from that updated repository.
!nvidia-smi -L
!pip -q install -r requirements-colab.txt

In [ ]:
# Kaggle: dùng Colab Secrets (KAGGLE_USERNAME, KAGGLE_KEY). KHÔNG viết khoá vào notebook.
import os
from google.colab import userdata
os.environ['KAGGLE_USERNAME'] = userdata.get('KAGGLE_USERNAME')
os.environ['KAGGLE_KEY'] = userdata.get('KAGGLE_KEY')

## 1. Dữ liệu: tải và chia split

In [ ]:
!python scripts/prepare_data.py --kaggle

In [ ]:
from student.data import validate_splits
summary = validate_splits()
print(summary['dataset_version'], summary['counts'])
assert summary['counts']['train']['images'] == 1422
assert summary['counts']['val']['images'] == 276
assert summary['counts']['test']['images'] == 311

## 2. Nhãn: YOLO → JSON thang 0–1000

In [ ]:
from student.data import objects_to_answer, answer_to_pixels, PROMPT
row = summary['entries']['train'][0]
answer = objects_to_answer(row['objects'])
print(PROMPT, answer)
print(answer_to_pixels(answer, row['width'], row['height']))

## 3. E0: zero-shot

In [ ]:
import subprocess
subprocess.run(['python', '-m', 'student.runner', '--exp', 'e0', '--mode', 'infer', '--split', 'val'], check=True)
subprocess.run(['python', '-m', 'student.runner', '--exp', 'e0', '--mode', 'infer', '--split', 'test'], check=True)

In [ ]:
!python scripts/score.py --pred work/e0.jsonl --name E0 --out report/evidence/e0.json

## 4. Dựng dữ liệu huấn luyện: chat template và mask loss

In [ ]:
from transformers import AutoProcessor
from student.data import encode_example, multimodal_collate
processor = AutoProcessor.from_pretrained('Qwen/Qwen3-VL-4B-Instruct')
sample = encode_example(processor, summary['entries']['train'][0], 'data')
print('input tokens:', sample['input_ids'].shape[1], 'loss tokens:', int((sample['labels'] != -100).sum()))
image_token = processor.tokenizer.convert_tokens_to_ids('<|image_pad|>')
print('image tokens:', int((sample['input_ids'] == image_token).sum()), 'patches:', sample['pixel_values'].shape[0], 'grid:', sample['image_grid_thw'].tolist())
assert int((sample['input_ids'] == image_token).sum()) == 220
assert int((sample['labels'] != -100).sum()) > 0

## 5. Setup LoRA / đóng băng cho E1, E2, E3

In [ ]:
from student.model import LLM_TARGETS, VIT_TARGETS
print('E1/E2 LLM target pattern:', LLM_TARGETS.pattern)
print('E3 ViT target pattern:', VIT_TARGETS.pattern)
print('The runner validates 252 matched modules for E1/E2 and 348 for E3 before training.')

## 6. Vòng huấn luyện

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
# One command per run. Resume uses saved optimizer/RNG state from Drive.
import subprocess
for exp in ('e1', 'e2', 'e3'):
    subprocess.run(['python', '-m', 'student.runner', '--exp', exp, '--mode', 'train', '--checkpoint-root', '/content/drive/MyDrive/day10'], check=True)

## 7. Suy luận và chấm E1–E3

In [ ]:
# Debug on validation first; inspect parse rate, F1 and samples. Test only after validation is satisfactory.
import subprocess
for exp in ('e1', 'e2', 'e3'):
    subprocess.run(['python', '-m', 'student.runner', '--exp', exp, '--mode', 'infer', '--split', 'val', '--checkpoint-root', '/content/drive/MyDrive/day10'], check=True)
for exp in ('e1', 'e2', 'e3'):
    subprocess.run(['python', '-m', 'student.runner', '--exp', exp, '--mode', 'infer', '--split', 'test', '--checkpoint-root', '/content/drive/MyDrive/day10'], check=True)

## 8. E4 (ablation tự chọn) và E5 (tuỳ chọn, GRPO)

In [ ]:
# E4 10%: same setup/hyperparameters/steps; deterministic 10% sample.
import subprocess
subprocess.run(['python', '-m', 'student.runner', '--exp', 'e4_data_10', '--mode', 'train', '--train-fraction', '0.1', '--checkpoint-root', '/content/drive/MyDrive/day10'], check=True)
subprocess.run(['python', '-m', 'student.runner', '--exp', 'e4_data_10', '--mode', 'infer', '--split', 'test', '--checkpoint-root', '/content/drive/MyDrive/day10'], check=True)
# E4 100% reuses the best E1/E2/E3 checkpoint selected from validation.
subprocess.run(['python', '-m', 'student.runner', '--exp', 'e4_data_100', '--mode', 'infer', '--split', 'test', '--checkpoint-root', '/content/drive/MyDrive/day10', '--base-exp', 'auto'], check=True)
subprocess.run(['python', '-m', 'student.reporting', '--make-figures', '--make-summary', '--compare-e4'], check=True)
# E5 GRPO bonus is intentionally kept for a separate later pass.